In [ ]:
from dataclasses import dataclass
from pathlib import Path

In [ ]:
1

2

In [10]:
import os
# os.chdir("../")
!pwd

/c/Users/Ibk/Desktop/data project/Stock-Price-Prediction-MLOps


In [31]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_path: Path
    mlflow_uri: str
    date_column: str
    params: list
    
    

In [32]:
from stock_prediction.constants import *
from stock_prediction.utils.common import *
from stock_prediction.entity.config_entity import *
import os
from dotenv import load_dotenv
mlflow_tracking_uri = load_dotenv("MLFLOW_TRACKING_URI")


class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir])
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            metric_file_path=config.metric_file_path,
            params=self.params.ARIMA.order,
            date_column=self.schema.date_column,
            mlflow_uri=mlflow_tracking_uri,
            
        )
        return model_evaluation_config

2026-08-12 04:45:47,869 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-12 04:45:47,872 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-12 04:45:47,875 | INFO | common| YAML file: schema.yaml loaded successfully.
2026-08-12 04:45:47,877 | INFO | common| Directory created at: artifacts
2026-08-12 04:45:47,879 | INFO | common| Directory created at: artifacts/model_evaluation


ModelEvaluationConfig(root_dir='artifacts/model_evaluation', test_data_path='artifacts/data_transformation/test.csv', model_path='artifacts/model_trainer/"arima.joblib"', metric_file_path='artifacts/model_evaluation/metric.json', mlflow_uri=False, date_column='date', params=BoxList([5, 2, 0]))

In [13]:
df = pd.read_csv(r"artifacts\data_transformation\test.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5348 entries, 0 to 5347
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    5348 non-null   object 
 1   close   5348 non-null   float64
dtypes: float64(1), object(1)
memory usage: 83.7+ KB


In [90]:
df = pd.read_csv(r"artifacts\data_ingestion\stock_data.csv", parse_dates=["date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6685 entries, 0 to 6684
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    6685 non-null   datetime64[ns]
 1   close   6685 non-null   float64       
 2   high    6685 non-null   float64       
 3   low     6685 non-null   float64       
 4   open    6685 non-null   float64       
 5   volume  6685 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 313.5 KB


In [76]:
df.set_index("date").close.loc[:"2026-05-31"]

date
2000-01-03      0.837002
2000-01-04      0.766434
2000-01-05      0.777650
2000-01-06      0.710353
2000-01-07      0.744002
                 ...    
2026-05-22    308.553894
2026-05-26    308.064301
2026-05-27    310.582153
2026-05-28    312.240723
2026-05-29    311.791107
Name: close, Length: 6641, dtype: float64

In [83]:
df.set_index("date").close[:"2026-04-30"].index.max()

'2026-04-30'

In [85]:
df.set_index("date").close["2026-04-30":].index.min()

'2026-04-30'

In [77]:
df.set_index("date").close.loc["2026-05-31":]

date
2026-06-01    306.046051
2026-06-02    314.928406
2026-06-03    309.992645
2026-06-04    310.961823
2026-06-05    307.075165
2026-06-08    301.280182
2026-06-09    290.299622
2026-06-10    291.328735
2026-06-11    295.375244
2026-06-12    290.879150
2026-06-15    296.164581
2026-06-16    298.982147
2026-06-17    295.694977
2026-06-18    297.753204
2026-06-22    296.754089
2026-06-23    294.046387
2026-06-24    292.827423
2026-06-25    274.912903
2026-06-26    283.535461
2026-06-29    281.497223
2026-06-30    289.110657
2026-07-01    294.126343
2026-07-02    308.364044
2026-07-06    312.390594
2026-07-07    310.392303
2026-07-08    313.119965
2026-07-09    315.947510
2026-07-10    315.048309
2026-07-13    317.036560
2026-07-14    314.588684
2026-07-15    327.217804
2026-07-16    332.972839
2026-07-17    333.452393
2026-07-20    326.308563
2026-07-21    327.457581
2026-07-22    325.609192
2026-07-23    321.382843
2026-07-24    332.733032
2026-07-27    336.619690
2026-07-28    339.78

In [22]:
df.set_index("date").head()

,close
date,
2005-04-29,1.078541
2005-05-02,1.089607
2005-05-03,1.083027
2005-05-04,1.111143
2005-05-05,1.097085


In [60]:

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from stock_prediction.utils.common import *
import mlflow
from urllib.parse import urlparse
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    def _eval_metrics(self, actual, pred):
        rmse = root_mean_squared_error(actual, pred)
        mae =  mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        test_data = test_data.set_index(self.config.date_column)
        arima_result = load_bin(Path(self.config.model_path))
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            predictions = arima_result.forecast(len(test_data))
            (rmse, mae, r2) = self._eval_metrics(test_data, predictions)
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(Path(self.config.metric_file_path), scores)
            mlflow.log_param("order", self.config.params)
            mlflow.log_metrics(metrics=scores)
            
            if tracking_uri_type_store != "file":
                mlflow.statsmodels.log_model(arima_result, name="model", registered_model_name="Arima_model")
            else:
                mlflow.statsmodels.log_model(arima_result, name="model")
        
        
        
        

In [61]:
def load_bin(path: Path) -> Any:
    """
    Loads a binary file using joblib and returns the data.
    Args:
        path (Path): Path to the binary file.
    Returns:
        Any: The data loaded from the binary file.
    """
    try:
        with open(path, "rb") as bin_file:
            data = joblib.load(bin_file)
        logger.info(f"Binary file loaded from: {path}")
        return data
    except Exception as e:
        logger.error(f"Error loading binary file from {path}: {e}")
        raise e


In [62]:
config = ConfigurationManager()
model_evaluation_config = config.get_model_evaluation_config()
model_evaluation = ModelEvaluation(model_evaluation_config)
model_evaluation.log_into_mlflow()


2026-08-12 05:09:25,419 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-12 05:09:25,422 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-12 05:09:25,425 | INFO | common| YAML file: schema.yaml loaded successfully.
2026-08-12 05:09:25,427 | INFO | common| Directory created at: artifacts
2026-08-12 05:09:25,429 | INFO | common| Directory created at: artifacts/model_evaluation
2026-08-12 05:09:25,500 | INFO | 592477671| Binary file loaded from: artifacts\model_trainer\arima.joblib
2026-08-12 05:09:25,671 | INFO | common| JSON file saved at: artifacts\model_evaluation\metric.json


c:\Users\Ibk\Desktop\data project\Stock-Price-Prediction-MLOps\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\Ibk\Desktop\data project\Stock-Price-Prediction-MLOps\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
Registered model 'Arima_model' already exists. Creating a new version of this model...
Created version '2' of model 'Arima_model'.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from dsProject.utils.common import *
from urllib.parse import urlparse
from dsProject.entity.config_entity import ModelEvaluationConfig
import mlflow

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        
    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = load_bin(Path(self.config.model_path))
        
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            predicted_qualites = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualites)
            
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(Path(self.config.metric_file_name), scores)
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)
            
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")
            
        